# Deep RL Trading Bot — Train on Colab
Single-asset BUY / HOLD / SELL agent (PPO via Stable-Baselines3) on yfinance data.

**Pipeline:** data (yfinance) → features (SMA, RSI, MACD) → Gymnasium env → PPO training → backtest vs Buy & Hold → save model.

Run the cells top to bottom. Set the runtime to GPU or CPU (CPU is fine for PPO with an MLP policy).

In [ ]:
!pip install -q stable-baselines3 gymnasium yfinance

## 1. Config

In [ ]:
TICKER      = "AAPL"
START       = "2012-01-01"
TRAIN_END   = "2021-12-31"   # train: START -> TRAIN_END
VAL_END     = "2022-12-31"   # validation: TRAIN_END -> VAL_END
END         = "2024-12-31"   # test: VAL_END -> END (never seen in training)

WINDOW      = 10             # past prices in the state (t-10 .. t)
INIT_CASH   = 10_000.0
FEE         = 0.001          # 0.1% transaction cost per trade
TOTAL_STEPS = 300_000        # PPO training timesteps
SEED        = 42

## 2. Download data

In [ ]:
import yfinance as yf, pandas as pd, numpy as np

raw = yf.download(TICKER, start=START, end=END, auto_adjust=True, progress=False)
if isinstance(raw.columns, pd.MultiIndex):          # newer yfinance returns multi-index columns
    raw.columns = raw.columns.get_level_values(0)
raw = raw[["Open","High","Low","Close","Volume"]].dropna()
print(raw.shape); raw.tail()

## 3. Feature engineering (SMA, RSI, MACD)

In [ ]:
def add_features(df):
    df = df.copy()
    c = df["Close"]
    df["ret"]      = c.pct_change()
    df["sma10"]    = c.rolling(10).mean()
    df["sma30"]    = c.rolling(30).mean()
    df["sma_ratio"] = df["sma10"] / df["sma30"] - 1          # trend signal

    delta = c.diff()
    gain  = delta.clip(lower=0).ewm(alpha=1/14, adjust=False).mean()
    loss  = (-delta.clip(upper=0)).ewm(alpha=1/14, adjust=False).mean()
    rs    = gain / (loss + 1e-9)
    df["rsi"] = (100 - 100 / (1 + rs)) / 100                 # scaled to 0-1

    ema12 = c.ewm(span=12, adjust=False).mean()
    ema26 = c.ewm(span=26, adjust=False).mean()
    macd  = ema12 - ema26
    signal = macd.ewm(span=9, adjust=False).mean()
    df["macd_delta"] = (macd - signal) / c                   # scale-free momentum

    return df.dropna()

data = add_features(raw)

train_df = data.loc[:TRAIN_END]
val_df   = data.loc[TRAIN_END:VAL_END]
test_df  = data.loc[VAL_END:]
print(len(train_df), len(val_df), len(test_df))

## 4. Gymnasium trading environment
- **Actions:** 0 = HOLD, 1 = BUY (invest all cash), 2 = SELL (liquidate)
- **State:** last 10 normalized prices + SMA ratio + RSI + MACD delta + cash fraction + position fraction + equity ratio
- **Reward:** `(V_{t+1} - V_t) / V_t` (portfolio % change), as in the slides

In [ ]:
import gymnasium as gym
from gymnasium import spaces

class TradingEnv(gym.Env):
    metadata = {"render_modes": []}

    def __init__(self, df, window=WINDOW, init_cash=INIT_CASH, fee=FEE):
        super().__init__()
        self.df = df.reset_index(drop=True)
        self.window, self.init_cash, self.fee = window, init_cash, fee
        self.prices = self.df["Close"].values.astype(np.float64)
        self.feats  = self.df[["sma_ratio","rsi","macd_delta"]].values.astype(np.float32)
        self.action_space = spaces.Discrete(3)
        obs_dim = window + 3 + 3
        self.observation_space = spaces.Box(-np.inf, np.inf, shape=(obs_dim,), dtype=np.float32)

    def _equity(self):
        return self.cash + self.shares * self.prices[self.t]

    def _obs(self):
        p = self.prices[self.t - self.window + 1 : self.t + 1]
        p_norm = (p / p[-1] - 1.0) * 10.0                    # relative to current price, scaled
        eq = self._equity()
        port = np.array([self.cash / eq,
                         self.shares * self.prices[self.t] / eq,
                         eq / self.init_cash - 1.0], dtype=np.float32)
        return np.concatenate([p_norm.astype(np.float32), self.feats[self.t], port])

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.t = self.window - 1
        self.cash, self.shares = self.init_cash, 0.0
        self.equity_curve = [self._equity()]
        self.trades = []                                     # (t, 'BUY'/'SELL', price)
        return self._obs(), {}

    def step(self, action):
        price = self.prices[self.t]
        v_prev = self._equity()

        if action == 1 and self.cash > 1e-6:                 # BUY all-in
            self.shares = self.cash * (1 - self.fee) / price
            self.cash = 0.0
            self.trades.append((self.t, "BUY", price))
        elif action == 2 and self.shares > 1e-9:             # SELL all
            self.cash = self.shares * price * (1 - self.fee)
            self.shares = 0.0
            self.trades.append((self.t, "SELL", price))

        self.t += 1
        terminated = self.t >= len(self.prices) - 1
        v_next = self._equity()
        reward = (v_next - v_prev) / v_prev
        self.equity_curve.append(v_next)
        return self._obs(), float(reward), terminated, False, {"equity": v_next}

# quick sanity check
from stable_baselines3.common.env_checker import check_env
check_env(TradingEnv(train_df))
print("Env OK")

## 5. Train PPO

In [ ]:
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize
from stable_baselines3.common.callbacks import EvalCallback

train_env = VecNormalize(DummyVecEnv([lambda: TradingEnv(train_df)]), norm_obs=False, norm_reward=True)
val_env   = DummyVecEnv([lambda: TradingEnv(val_df)])

model = PPO(
    "MlpPolicy", train_env,
    learning_rate=3e-4, n_steps=1024, batch_size=128, gamma=0.99,
    gae_lambda=0.95, ent_coef=0.01, clip_range=0.2,
    policy_kwargs=dict(net_arch=[128, 128]),
    seed=SEED, verbose=1,
)

eval_cb = EvalCallback(val_env, best_model_save_path="./best_model", eval_freq=10_000,
                       n_eval_episodes=1, deterministic=True, verbose=1)
model.learn(total_timesteps=TOTAL_STEPS, callback=eval_cb)
model.save("ppo_trading_final")

## 6. Backtest on unseen test data vs Buy & Hold

In [ ]:
import os
best_path = "./best_model/best_model.zip"
agent = PPO.load(best_path if os.path.exists(best_path) else "ppo_trading_final")

def run_backtest(model, df):
    env = TradingEnv(df)
    obs, _ = env.reset()
    done = False
    while not done:
        action, _ = model.predict(obs, deterministic=True)
        obs, r, done, _, _ = env.step(int(action))
    return env

env_test = run_backtest(agent, test_df)
agent_curve = np.array(env_test.equity_curve)

# Buy & Hold baseline
prices = test_df["Close"].values[WINDOW-1:]
bh_curve = INIT_CASH * (prices / prices[0])
n = min(len(agent_curve), len(bh_curve))
agent_curve, bh_curve = agent_curve[:n], bh_curve[:n]

def metrics(curve):
    rets = np.diff(curve) / curve[:-1]
    total = curve[-1] / curve[0] - 1
    sharpe = np.sqrt(252) * rets.mean() / (rets.std() + 1e-9)
    peak = np.maximum.accumulate(curve)
    mdd = ((curve - peak) / peak).min()
    return {"Total Return %": round(total*100,2), "Sharpe": round(sharpe,2), "Max Drawdown %": round(mdd*100,2)}

res = pd.DataFrame({"RL Agent": metrics(agent_curve), "Buy & Hold": metrics(bh_curve)})
print(res)
print("Number of trades:", len(env_test.trades))

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(2, 1, figsize=(12, 8), sharex=True)
dates = test_df.index[WINDOW-1:][:n]

ax[0].plot(dates, agent_curve, label="RL Agent")
ax[0].plot(dates, bh_curve, label="Buy & Hold", alpha=0.7)
ax[0].set_title(f"{TICKER} - Portfolio Value (Test Period)"); ax[0].legend(); ax[0].grid(alpha=.3)

ax[1].plot(dates, test_df["Close"].values[WINDOW-1:][:n], color="gray", lw=1)
for t, side, p in env_test.trades:
    idx = t - (WINDOW-1)
    if 0 <= idx < n:
        ax[1].scatter(dates[idx], p, marker="^" if side=="BUY" else "v",
                      color="green" if side=="BUY" else "red", s=60)
ax[1].set_title("Price with BUY (green) / SELL (red) markers"); ax[1].grid(alpha=.3)
plt.tight_layout(); plt.savefig("backtest.png", dpi=150); plt.show()

## 7. Save & download
Downloads the trained model, the config and the plot so you can use them inside your Antigravity project (`models/` folder).

In [ ]:
import json, shutil
shutil.copy(best_path if os.path.exists(best_path) else "ppo_trading_final.zip", "ppo_trading_best.zip")
json.dump({"ticker": TICKER, "window": WINDOW, "init_cash": INIT_CASH, "fee": FEE,
           "features": ["sma_ratio","rsi","macd_delta"]}, open("config.json","w"))
res.to_csv("test_metrics.csv")

from google.colab import files
for f in ["ppo_trading_best.zip", "config.json", "test_metrics.csv", "backtest.png"]:
    files.download(f)

## Optional: multi-seed check
RL results vary a lot with the random seed. Re-run training with `SEED = 1, 2, 3` and report the **average** test metrics for a more honest result.